# Woche 6: Wiener-Filter zur Entrauschung

Diese Woche behandelt den Wiener-Filter. Er arbeitet im Spektralbereich und versucht, Rauschen zu unterdruecken, ohne das Nutzsignal zu stark zu beschaedigen.

## Lernziele

Nach dieser Woche sollten Sie:

- das additive Rauschmodell erklaeren koennen
- den Wiener-Filter als frequenzabhaengiges Gewicht verstehen
- Leistungsspektren von Betragsspektren unterscheiden koennen
- verstehen, warum Rauschen geschaetzt werden muss
- stationaeres Rauschen erklaeren koennen
- nicht-kausale und kausale Schaetzung unterscheiden koennen
- die Funktion `FilterEstimator2` aus dem Notebook nachvollziehen koennen

## Additives Rauschmodell

Das verrauschte Signal wird modelliert als:

$y(n)=x(n)+d(n)$

mit:

- $x(n)$: originales Nutzsignal
- $d(n)$: Rauschen
- $y(n)$: gemessenes verrauschtes Signal

Im Frequenzbereich gilt entsprechend:

$Y(f)=X(f)+D(f)$

Wenn Nutzsignal und Rauschen unkorreliert sind, kann fuer die Leistung grob angenommen werden:

$|Y(f)|^2 \approx |X(f)|^2 + |D(f)|^2$

## Idee des Wiener-Filters

Der Wiener-Filter ist ein Gewicht $H(f)$ zwischen 0 und 1.

$Z(f)=Y(f)\cdot H(f)$

- Wenn eine Frequenz wahrscheinlich viel Nutzsignal enthaelt, soll $H(f)$ nahe 1 sein.
- Wenn eine Frequenz wahrscheinlich viel Rauschen enthaelt, soll $H(f)$ nahe 0 sein.

So wird das verrauschte Spektrum frequenzweise abgeschwaecht.

## Grundformel

Eine einfache Form des Wiener-Filters lautet:

$H(f)=\frac{|X(f)|^2}{|X(f)|^2+|D(f)|^2}$

Interpretation:

- Zaehler: Leistung des Nutzsignals
- Nenner: Gesamtleistung aus Nutzsignal und Rauschen
- Ergebnis: Anteil, der als Nutzsignal interpretiert wird

Wenn $|X(f)|^2$ gross ist und $|D(f)|^2$ klein, wird $H(f)$ gross. Wenn das Rauschen dominiert, wird $H(f)$ klein.

In [ ]:
import numpy as np

signal_power = np.array([10.0, 1.0, 0.1])
noise_power = np.array([1.0, 1.0, 1.0])
H = signal_power / (signal_power + noise_power)
H

## Warum muss Rauschen geschaetzt werden?

In einer echten Messung kennt man normalerweise nur $y(n)$. Das reine Signal $x(n)$ und das reine Rauschen $d(n)$ sind unbekannt.

Deshalb kann die ideale Formel nicht direkt verwendet werden. Man braucht eine Schaetzung fuer das Rauschen.

Eine einfache Annahme: Das Rauschen ist stationaer. Das bedeutet, dass sein Spektrum ueber die Zeit ungefaehr gleich bleibt.

## Rauschschaetzung aus dem Spektrogramm

Im Notebook wird das Rauschen aus dem verrauschten Spektrogramm $Y(f,t)$ geschaetzt. Fuer jede Frequenz wird ueber die Zeit gemittelt:

$|D(f)| \approx \sqrt{\frac{1}{T}\sum_{t=0}^{T-1}|Y(f,t)|^2}$

Das ergibt eine Rauschschaetzung pro Frequenz. Diese wird anschliessend auf alle Zeitspalten kopiert, damit sie dieselbe Form wie das Spektrogramm hat.

## `FilterEstimator2` einfach erklaert

Die Funktion aus dem Notebook:

```python
def FilterEstimator2(X, Y):
    D = np.sqrt(np.mean(Y**2, axis = 1))
    D = np.outer(D, np.ones(Y.shape[1]))
    H = 1 - (D**2 + 1e-16) / (Y**2 + 1e-16)
    H = np.maximum(H, 0.0)
    H = np.minimum(H, 1.0)
    return H
```

Bedeutung:

- `Y` ist das Betragsspektrogramm des verrauschten Signals.
- `D` schaetzt das Rauschen pro Frequenz.
- `H` ist der Filter zwischen 0 und 1.
- `np.maximum` und `np.minimum` begrenzen den Filter auf sinnvolle Werte.
- `1e-16` verhindert Division durch 0.

In [ ]:
Y = np.array([
    [1.0, 2.0, 1.5],
    [0.2, 0.1, 0.3]
])

D = np.sqrt(np.mean(Y**2, axis=1))
D_matrix = np.outer(D, np.ones(Y.shape[1]))
H = 1 - (D_matrix**2 + 1e-16) / (Y**2 + 1e-16)
H = np.maximum(H, 0.0)
H = np.minimum(H, 1.0)

D, D_matrix, H

## Warum wird begrenzt?

Die Formel kann negative Werte liefern, wenn die Rauschschaetzung groesser als der gemessene Wert ist. Ein negativer Filter waere hier nicht sinnvoll.

Deshalb:

- Werte kleiner als 0 werden auf 0 gesetzt.
- Werte groesser als 1 werden auf 1 gesetzt.

Damit bleibt $H(f,t)$ ein verstaendliches Gewicht: 0 bedeutet unterdruecken, 1 bedeutet durchlassen.

## Feinabstimmung mit Alpha

Im Notebook wird eine Variante eingefuehrt:

$H(f)=1-\left(\frac{|D(f)|^2}{|Y(f)|^2}\right)^\alpha$

Der Parameter $\alpha$ steuert, wie stark der Filter eingreift.

- $\alpha=1$: normale Variante
- $\alpha=\frac{1}{2}$: entspricht eher einer Spektralsubtraktion

Ein Random Walk sucht nach einem guten Wert fuer $\alpha$, indem verschiedene Werte ausprobiert und anhand des SNR verglichen werden.

## Hoelder-Mittel fuer die Rauschschaetzung

Statt immer den quadratischen Mittelwert zu verwenden, kann ein allgemeineres Mittel verwendet werden:

$\|D(f)\|_p=\left(\frac{1}{T}\sum_{t=0}^{T-1}|Y(f,t)|^p\right)^{1/p}$

Der Parameter $p$ veraendert, wie stark grosse oder kleine Werte in die Schaetzung eingehen. Auch dieser Parameter kann optimiert werden.

## Kausal vs. nicht-kausal

Ein nicht-kausaler Filter darf auch zukuenftige Spektrogrammspalten verwenden. Das ist bei gespeicherten Audiodateien moeglich, aber nicht in Echtzeit.

Ein kausaler Filter darf nur aktuelle und vergangene Informationen verwenden.

In Echtzeitanwendungen ist Kausalitaet wichtig, weil zukuenftige Samples noch nicht bekannt sind.

In [ ]:
# Kausale Idee: Fuer jede Spalte nur aktuelle und vergangene Spalten verwenden.
Y = np.abs(np.random.randn(4, 6))
H = np.zeros_like(Y)

for t in range(Y.shape[1]):
    past = Y[:, :t + 1]
    D = np.sqrt(np.mean(past**2, axis=1))
    D = np.outer(D, np.ones(1)).reshape(-1)
    H[:, t] = 1 - (D**2 + 1e-16) / (Y[:, t]**2 + 1e-16)

H = np.maximum(H, 0.0)
H = np.minimum(H, 1.0)
H

## SNR als Bewertung

Das Signal-Rausch-Verhaeltnis beschreibt, wie stark das Nutzsignal im Vergleich zum Fehler/Rauschen ist:

$SNR = 10\log_{10}\left(\frac{P_\text{Signal}}{P_\text{Rauschen}}\right)$

Ein hoeheres SNR bedeutet normalerweise bessere Entrauschung. Aber ein Filter kann das SNR verbessern und trotzdem hoerbare Artefakte erzeugen. Deshalb ist SNR hilfreich, aber nicht die einzige Qualitaetsbewertung.

## Typische Fehler

- `Y` als Zeitsignal statt als Spektrogramm interpretieren.
- Betrag, Leistung und komplexes Spektrum verwechseln.
- `axis=1` falsch verstehen: Es wird ueber Zeitspalten gemittelt.
- Nicht-kausale Verfahren fuer Echtzeit verwenden.
- Division durch 0 nicht absichern.
- Einen Filter ausserhalb des Bereichs 0 bis 1 nicht begrenzen.

## Mini-Check

1. Was bedeutet $y(n)=x(n)+d(n)$?
2. Warum liegt der Wiener-Filter typischerweise zwischen 0 und 1?
3. Warum ist eine Rauschschaetzung notwendig?
4. Was bedeutet stationaeres Rauschen?
5. Was ist der Unterschied zwischen kausal und nicht-kausal?
6. Warum wird `1e-16` in den Nenner addiert?
7. Was macht `axis=1` bei `np.mean(Y**2, axis=1)`?